# filter / where — try it live

Runnable companion to the note: **[filter / where](https://ravi-writes.pages.dev/notes/abinitio-to-pyspark/core-operations/filter)**.

`filter` (alias `where`) keeps only the rows where a boolean condition is **true** — the equivalent of Ab Initio's **Filter By Expression**. Run the cells top to bottom; nothing is installed on your machine.

## The Ab Initio equivalent

In an Ab Initio graph this is **Create Data → Filter By Expression → Trash**. The **Filter By Expression (FBE)** component has two output ports: **`select`** carries the rows where the condition is *true* (what this example keeps and shows), and **`deselect`** carries the rest — the rows that *did not match* the condition.

> **Not the same as `reject`.** Ab Initio's `reject` port is for records that *errored* and couldn't be processed. A `deselect` row is perfectly valid — it just tested false.

Both flows end in **Trash** here because we're only inspecting behaviour, not persisting anything.

```text
  Create Data
       │
       ▼
  Filter By Expression   (balance > 0 AND name is not null)
       │
       ├─ select    ──►  Trash    (kept — condition true)
       └─ deselect  ──►  Trash    (did not match — condition false)
```


## 1. Install & start Spark

`SparkSession.builder...getOrCreate()` is the builder object — see [The Builder Object](https://ravi-writes.pages.dev/notes/python-for-spark/).

In [ ]:
!pip install -q pyspark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.appName("filter-demo").getOrCreate()

## 2. Build a tiny DataFrame (inline data — no file needed)

In [ ]:
df = spark.createDataFrame(
    [
        (1, "Asha", "2024-03-01", 120.0),
        (2, "Ben",  "2023-11-05",   0.0),
        (3, None,   "2024-06-20",  45.0),
    ],
    ["id", "name", "signup_date", "balance"],
)

df.show()

## 3. Filter — keep positive balance AND non-null name

In [ ]:
kept = df.filter((col("balance") > 0) & col("name").isNotNull())
kept.show()

# Row 2 drops (balance not > 0); row 3 drops (name is null).

## Your turn

1. **Capture what didn't match** — Ab Initio's `deselect` port. Filter keeps only matches, so use the negated condition.
2. **Same result, SQL style** — rewrite the keep-condition as a single SQL string with `df.where(...)`.
3. **Watch nulls disappear** — filter on `balance > 100` alone and note that the null-name row still drops out.

Try them yourself first, then reveal the solutions below.

In [ ]:
# 1. What didn't match — the negated condition (the 'deselect' port)
unmatched = df.filter(~((col("balance") > 0) & col("name").isNotNull()))
unmatched.show()

# 2. Same keep-result, SQL string
df.where("balance > 0 AND name IS NOT NULL").show()

# 3. Nulls are not true -> the null-balance row (row... none here) and
#    any null comparison drops, exactly like SQL WHERE
df.filter(col("balance") > 100).show()